### 创建智能体

1. 安装依赖(包括LangChain、LangGraph、DeepAgents)
2. 初始化LLM
3. 初始化工具列表
4. 初始化智能体
5. 测试智能体

1. 安装依赖

```
pip install -U langchain langgraph deepagents
```

目前langchain和langgraph要求python版本号<strong>3.10及3.10+</strong>,
deepagents要求python版本号<strong>3.10+</strong>,
所以推荐在python=3.11环境下进行相关内容的学习

In [ ]:
%pip install -U langchain langchain_openai langgraph deepagents

2. 初始化LLM

通常在本地环境或.env配置LLM相关参数

- 在本地环境进行配置

**Windows**环境下：

```bash
# 临时(当前终端窗口生效)
set MODEL_NAME=deepseek-flash
set MODEL_API_KEY=sk-...
set MODEL_API_URL=https://api.deepseek.com/v1

# 永久(powershell，写入用户环境变量，重启终端生效)
[Environment]::SetEnvironmentVariable("MODEL_NAME", "deepseek-flash", "User")
[Environment]::SetEnvironmentVariable("MODEL_API_KEY", "sk-...", "User")
[Environment]::SetEnvironmentVariable("MODEL_API_URL", "https://api.deepseek.com/v1", "User")
```

**Linux**环境下：

```bash
# 临时
export MODEL_NAME="deepseek-flash"
export MODEL_API_KEY="sk-..."
export MODEL_API_URL="https://api.deepseek.com/v1"

# 永久
echo 'export MODEL_NAME="deepseek-flash"' >> ~/.bashrc
echo 'export MODEL_API_KEY="sk-..."' >> ~/.bashrc
echo 'export MODEL_API_URL="https://api.deepseek.com/v1"' >> ~/.bashrc
source ~/.bashrc
```

- 使用.env文件进行配置

```text
# 示例
MODEL_NAME="deepseek-flash"
MODEL_API_KEY=sk-...
MODEL_API_URL=https://api.deepseek.com/v1
```

**读取环境变量**

```python
import os
# %pip install python_dotenv
from dotenv import load_dotenv

load_dotenv()

model_name = os.getenv("MODEL_NAME")
api_key = os.getenv("MODEL_API_KEY")
api_url = os.getenv("MODEL_API_URL")
```

In [ ]:
# 在Windows环境下临时配置LLM相关参数
# %env是jupyter原生魔法，Winodws终端环境下应该使用set
%env MODEL_NAME=deepseek-flash
# 需要替换为真实API Key
%env MODEL_API_KEY=sk-...
%env MODEL_API_URL=https://api.deepseek.com/v1

In [ ]:
%pip install dotenv

In [32]:
import os
from dotenv import load_dotenv
from langchain.chat_models import init_chat_model

load_dotenv()

model_name = os.getenv("MODEL_NAME")
model_api_key = os.getenv("MODEL_API_KEY")
model_api_url = os.getenv("MODEL_API_URL")

# 其他参数可以从https://reference.langchain.com/python/langchain/chat_models/base/init_chat_model查阅
chat_model = init_chat_model(
    model=model_name,
    model_provider="openai", # deepseek兼容openai协议
    api_key=model_api_key,
    base_url=model_api_url,
    temperature=0.1 #随机性，越接近1随机性越强
)

# 测试
# chat_model 输入消息列表
ai_reply = chat_model.invoke([
        {
            "role":"user",
            "content":"你好"
        }
    ])

print(ai_reply)
print(ai_reply.content)

content='你好！有什么我可以帮你的吗？' additional_kwargs={'refusal': None} response_metadata={'token_usage': {'completion_tokens': 46, 'prompt_tokens': 31, 'total_tokens': 77, 'completion_tokens_details': {'accepted_prediction_tokens': None, 'audio_tokens': None, 'reasoning_tokens': 37, 'rejected_prediction_tokens': None, 'text_tokens': None}, 'prompt_tokens_details': {'audio_tokens': None, 'cache_write_tokens': None, 'cached_tokens': 0, 'image_tokens': None, 'text_tokens': None}, 'prompt_cache_hit_tokens': 0, 'prompt_cache_miss_tokens': 31}, 'model_provider': 'openai', 'model_name': 'deepseek-flash', 'system_fingerprint': 'aeb56401ca74e127821c4f9126dcb669', 'id': 'd0d32de9-b913-4308-91b0-c9973df33cc5', 'finish_reason': 'stop', 'logprobs': None} id='lc_run--01a13337-e82a-7701-8a14-dc95428a1e4e-0' tool_calls=[] invalid_tool_calls=[] usage_metadata={'input_tokens': 31, 'output_tokens': 46, 'total_tokens': 77, 'input_token_details': {'cache_read': 0}, 'output_token_details': {'reasoning': 37}}
你好！有什么我可

3. 初始化工具列表

工具重要参数：

- name
- description
- args_schema

**@tool**作用：

1. 生成tool description

将函数的文档字符串""" """变为工具描述。数据类型是string。

2. 生成tool args_schema

读取函数的type hint,生成符合OpenAI function calling的参数结构。

3. 将普通函数变为Base Tool对象

包装后函数拥有.name、.description、.args_schema属性，传给bind_tools或者create_agent。

4. 适配LangGraph的ToolNode

LangGraph中的ToolNode会自动识别@tool装饰的工具，
自动执行、把结果包装成ToolMessage,
不需要处理消息转换

In [33]:
from langchain_core.tools import tool
from langchain_core.messages import HumanMessage

@tool
def get_weather(city: str) -> str:
    """返回该城市的天气情况
    
    Args:
        city: 城市名

    Returns:
        str: 该城市的天气情况，例如"晴天"
    """
    return f"{city}当前天气：晴天"

chat_model_with_tools = chat_model.bind_tools([get_weather])

res = chat_model_with_tools.invoke([HumanMessage(content="贵阳天气怎么样？")])

print("res.tool_calls:", res.tool_calls)

res.tool_calls: [{'name': 'get_weather', 'args': {'city': '贵阳'}, 'id': 'call_00_a29QQS5WGWYb2oCUi2HJ9437', 'type': 'tool_call'}]


**chat_model**通过
*bind_tools*绑定tool list后，当识别到存在工具调用的需求，会返回tool_calls表示需要调用工具，chat_model没有工具调用的能力，因为其只是LLM，不是Agent

4. 初始化智能体

不同于
**chat_model**,
**agent**拥有了工具调用的能力,遵守ReAct模式,
即reasoning->action->observation的不断循环，
具体可查阅论文[ReAct: Synergizing Reasoning and Acting in Language Models](https://arxiv.org/abs/2210.03629)

In [34]:
from langchain.agents import create_agent
from langchain_core.messages import HumanMessage

agent = create_agent(
    model=chat_model,
    tools=[get_weather],
    system_prompt="You are a helpful assistant"
)

In [35]:
# 5. 测试智能体

res = agent.invoke(
    {
        "messages":[
            HumanMessage(content="贵阳天气怎么样？")
        ]
    }
)

print(res["messages"][-1].content_blocks)

[{'type': 'text', 'text': '贵阳当前天气是**晴天** ☀️，适合出门活动。需要我帮你查一下其他城市或者未来几天的天气吗？'}]
